%md
# Ingestão de Dados Silver - Ambiente de Produção

Iremos replicar o mesmo código 3_1. Apenas alterando o catálogo para o prod.

In [0]:
# ===================================================================
# CAMADA SILVER: TRANSFORMAÇÃO E ENRIQUECIMENTO DE DADOS (PySpark)
# ===================================================================
# Empresa: AluMax - Atendimento Omnichannel
# Objetivo: Criar tabela silver_atendimentos com parsing de JSON e 
#           padronização de dados a partir da camada Bronze usando PySpark
# ===================================================================

# PASSO 1: Criação do Schema Silver
# Garante que o database silver existe antes de criar as tabelas

spark.sql("""
    CREATE DATABASE IF NOT EXISTS prod.silver
    COMMENT 'Schema da Camada Silver - Dados limpos e enriquecidos'
""")

print("✓ Schema 'prod.silver' criado/verificado com sucesso!")

In [0]:
# ===================================================================
# PASSO 2: Leitura dos Dados da Camada Bronze
# ===================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, BooleanType, DoubleType, TimestampType

# Lê a tabela Bronze
df_bronze = spark.table("prod.bronze.atendimentos_alumax")

# ===================================================================
# PASSO 3: Transformação e Parsing do JSON
# ===================================================================

# IMPORTANTE: O payload_detalhes vem com aspas extras e aspas internas duplicadas
# Exemplo: "{""browser"": ""Safari"", ...}"
# Precisamos limpar isso primeiro antes de fazer o parsing

# Criar coluna com payload limpo
df_bronze_clean = df_bronze.withColumn(
    "payload_limpo",
    F.regexp_replace(
        F.regexp_replace(
            F.col("payload_detalhes"),
            '^"|"\'$',  # Remove aspas do início e fim
            ''
        ),
        '""',  # Substitui aspas duplas por aspas simples
        '"'
    )
)

# Aplicar transformações básicas e parsing condicional do JSON
df_silver = df_bronze_clean.select(
    # ===================================================================
    # COLUNAS BÁSICAS (padronizadas)
    # ===================================================================
    F.col("id_interacao"),
    F.col("cliente_id"),
    
    # Padronização de texto: minúsculas para canal e status
    F.lower(F.col("canal")).alias("canal"),
    F.lower(F.col("status")).alias("status"),
    
    # Padronização de texto: iniciais maiúsculas para departamento
    F.initcap(F.col("departamento")).alias("departamento"),
    
    # Conversão explícita para TIMESTAMP
    F.col("data_hora").cast(TimestampType()).alias("data_hora"),
    
    F.col("data_particao").alias("data_particao"),
    # ===================================================================
    # PARSING JSON - CAMPOS ESPECÍFICOS DO WHATSAPP
    # ===================================================================
    F.when(
        F.lower(F.col("canal")) == "whatsapp",
        F.get_json_object(F.col("payload_limpo"), "$.numero_origem")
    ).alias("whatsapp_numero_origem"),
    
    F.when(
        F.lower(F.col("canal")) == "whatsapp",
        F.get_json_object(F.col("payload_limpo"), "$.atendente_bot").cast(BooleanType())
    ).alias("whatsapp_atendente_bot"),
    
    F.when(
        F.lower(F.col("canal")) == "whatsapp",
        F.get_json_object(F.col("payload_limpo"), "$.tempo_resposta_bot_seg").cast(IntegerType())
    ).alias("whatsapp_tempo_resposta_bot_seg"),
    
    F.when(
        F.lower(F.col("canal")) == "whatsapp",
        F.get_json_object(F.col("payload_limpo"), "$.versao_whatsapp_api")
    ).alias("whatsapp_versao_api"),
    
    # ===================================================================
    # PARSING JSON - CAMPOS ESPECÍFICOS DO TELEFONE
    # ===================================================================
    F.when(
        F.lower(F.col("canal")) == "telefone",
        F.get_json_object(F.col("payload_limpo"), "$.duracao_chamada_seg").cast(IntegerType())
    ).alias("telefone_duracao_chamada_seg"),
    
    F.when(
        F.lower(F.col("canal")) == "telefone",
        F.get_json_object(F.col("payload_limpo"), "$.fila_espera_seg").cast(IntegerType())
    ).alias("telefone_fila_espera_seg"),
    
    F.when(
        F.lower(F.col("canal")) == "telefone",
        F.get_json_object(F.col("payload_limpo"), "$.protocolo")
    ).alias("telefone_protocolo"),
    
    F.when(
        F.lower(F.col("canal")) == "telefone",
        F.get_json_object(F.col("payload_limpo"), "$.transferencias").cast(IntegerType())
    ).alias("telefone_transferencias"),
    
    # ===================================================================
    # PARSING JSON - CAMPOS ESPECÍFICOS DO EMAIL
    # ===================================================================
    F.when(
        F.lower(F.col("canal")) == "email",
        F.get_json_object(F.col("payload_limpo"), "$.dominio_email")
    ).alias("email_dominio"),
    
    F.when(
        F.lower(F.col("canal")) == "email",
        F.get_json_object(F.col("payload_limpo"), "$.tamanho_corpo_bytes").cast(IntegerType())
    ).alias("email_tamanho_corpo_bytes"),
    
    F.when(
        F.lower(F.col("canal")) == "email",
        F.get_json_object(F.col("payload_limpo"), "$.anexos_quantidade").cast(IntegerType())
    ).alias("email_anexos_quantidade"),
    
    F.when(
        F.lower(F.col("canal")) == "email",
        F.get_json_object(F.col("payload_limpo"), "$.tempo_primeira_resposta_horas").cast(DoubleType())
    ).alias("email_tempo_primeira_resposta_horas"),
    
    # ===================================================================
    # PARSING JSON - CAMPOS ESPECÍFICOS DO CHAT SITE
    # ===================================================================
    F.when(
        F.lower(F.col("canal")) == "chat_site",
        F.get_json_object(F.col("payload_limpo"), "$.browser")
    ).alias("chat_browser"),
    
    F.when(
        F.lower(F.col("canal")) == "chat_site",
        F.get_json_object(F.col("payload_limpo"), "$.ip_origem")
    ).alias("chat_ip_origem"),
    
    F.when(
        F.lower(F.col("canal")) == "chat_site",
        F.get_json_object(F.col("payload_limpo"), "$.pagina_origem")
    ).alias("chat_pagina_origem"),
    
    F.when(
        F.lower(F.col("canal")) == "chat_site",
        F.get_json_object(F.col("payload_limpo"), "$.satisfacao_pre_atendimento").cast(IntegerType())
    ).alias("chat_satisfacao_pre_atendimento"),
    
    # ===================================================================
    # METADADOS DE AUDITORIA
    # ===================================================================
    # Preserva o timestamp de ingestão original da Bronze
    F.col("_ingestion_timestamp").alias("bronze_ingestion_timestamp"),
    
    # Adiciona timestamp de processamento na Silver
    F.current_timestamp().alias("silver_processing_timestamp"),
    
    # Preserva informação do arquivo fonte
    F.col("_source_file")
)

print("✓ Transformações aplicadas com sucesso!")
print(f"\nTotal de colunas na Silver: {len(df_silver.columns)}")
print("\nSchema da tabela Silver:")
df_silver.printSchema()

# ===================================================================
# PASSO 4: Gravação da Tabela Silver
# ===================================================================

# Salva o DataFrame como tabela Delta no catálogo Unity Catalog
# Usa a opção partitionOverwriteMode diretamente no write para compatibilidade com Serverless
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("partitionOverwriteMode", "dynamic") \
    .partitionBy("data_particao") \
    .saveAsTable("prod.silver.silver_atendimentos")

print("✓ Tabela 'prod.silver.silver_atendimentos' criada com sucesso!")
print(f"\nTotal de registros gravados: {df_silver.count():,}")


In [0]:
%sql
select * from prod.silver.silver_atendimentos